# Search or Think? Reproduce the evaluation results

**Runtime:** any Colab GPU (Runtime, Change runtime type). The free **T4** works: the policies then run in float16, because the T4 has no bfloat16. Every result in the report was produced on an **NVIDIA L4** in bfloat16 with vLLM 0.31.0 and TRL 1.16.0, and this notebook installs the same versions, so an L4 (Colab's paid tiers) gives the closest match. No Google Drive is needed, and no training is run.

**Time:** about one hour on an L4. Our own run took 56 minutes: 14 to set up the search index, 8 to download the checkpoints, and 33 to evaluate, plus a few minutes to install the libraries and 2 for step 6. A T4 is slower.

**What it does.** Runs `reproduce.py` from the repository:

1. checks the GPU and the library versions, and picks bfloat16 if the GPU has it, else float16;
2. runs the unit tests (`test_sot.py`, `test_export_site.py`, `test_reproduce.py`);
3. starts the BM25 search server over Pyserini's prebuilt `wikipedia-dpr-100w` index, then searches 2,000 queries from the recorded runs again and checks that the same top 3 passages come back;
4. downloads the untrained Qwen2.5-0.5B-Instruct from the Hugging Face Hub and the five trained policies from this repository's release (each checked against its SHA-256): the start policy (stage A2), plain GRPO, fixed penalty (lambda 0.05), SAAS and Soft-SAAS, seed 0 each;
5. evaluates every policy with greedy decoding on the five test sets (HotpotQA 1,000, NQ 500, 2WikiMultihopQA 1,000, MuSiQue 500, PopQA 500 questions), with the same commands as the original evaluation;
6. evaluates the start policy a second time, to measure how much two runs of the same command differ;
7. compares every policy on every test set with the recorded run, and prints **PASS** or **FAIL**.

**Why answers can differ between runs.** vLLM does not make greedy decoding bit for bit repeatable by default ([vLLM docs](https://docs.vllm.ai/en/v0.31.0/usage/reproducibility/)): requests are batched differently from run to run, so a few uncertain answers change, and a changed search query changes the rest of its episode. The original evaluation ran the same way. Step 6 shows the effect directly: in our run, two runs of the same command agreed on 79% of HotpotQA answers, while their F1 differed by only 0.4 points. In float16 (on a T4) the arithmetic itself differs too, so more answers change; the same rule judges the numbers.

**How a result is judged**, per policy and test set: the paired bootstrap 95% interval of the F1 difference (reproduced minus recorded, over the same questions, 2,000 resamples, as in the report) must lie within ±2.5 F1 points, and the search count within 0.1. Two runs of the same command gave intervals reaching about 2 points, so 2.5 allows that noise; seeds of the same reward differ by up to 4 F1 points. The two base model baselines without recorded answers (no search, RAG) must instead be within 1 point of the report's table. The retrieval check must return the recorded passages for at least 99% of queries. Anything else is a FAIL.

## 1. Get the code

In [ ]:
!git clone -q --depth 1 https://github.com/Kodezilla0725/search_or_think.git /content/search_or_think 2>/dev/null || git -C /content/search_or_think pull -q
!git -C /content/search_or_think log -1 --format="code at commit %h, %cd"

## 2. Install the versions the evaluation used

In [ ]:
!pip install -q "trl[vllm,peft]==1.16.0" "vllm==0.31.0" pytest
!pip uninstall -y -q torchao

## 3. Run the pipeline

Prints one line per step, the summary line of every evaluation, then the full results table and the verdict. The detailed log of each evaluation is in `/content/repro/logs/`.

In [ ]:
!cd /content/search_or_think && python -u reproduce.py --work /content/repro

## 4. Results

In [ ]:
import json

report = json.load(open("/content/repro/reproduce_report.json"))
print("VERDICT:", report["verdict"])
print("GPU:", report.get("environment", {}).get("gpu"), "| precision:", report.get("dtype"))
print("unit tests:", report.get("unit_tests", {}).get("summary"))
print("retrieval check:", {k: v for k, v in report.get("retrieval", {}).items() if k != "examples"})
print("two runs of the same command:", report.get("noise"))
print("minutes:", report["minutes"].get("total"))
rows = [r for r in report["rows"] if r["status"] != "not run"]
print(f"{sum(r['status'] == 'same' for r in rows)} of {len(rows)} policy and test set results are the same as recorded")
for r in rows:
    if r["status"] != "same":
        print(r)